### Imports and Data imports

In [1]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA version: {torch.version.cuda}")
print(f"Supported archs: {torch.cuda.get_arch_list()}")
print(f"GPU capability: {torch.cuda.get_device_capability(0)}")

PyTorch version: 2.12.0.dev20260401+cu128
CUDA version: 12.8
Supported archs: ['sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120']
GPU capability: (12, 0)


In [2]:
import os
from pathlib import Path
import random
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split

import matplotlib.pyplot as plt

In [3]:
# ---- paths ----
DATA_ROOT = Path("/cluster/courses/cil/monocular-depth-estimation/train")

# ---- training config ----
IMG_SIZE = 192          # slightly higher resolution gives the model more geometric detail
BATCH_SIZE = 6
NUM_EPOCHS = 8
LR = 5e-4
MAX_TRAIN_SAMPLES = 4000
MAX_VAL_SAMPLES = 800
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", DEVICE)

Using device: cuda


In [ ]:
class SimpleDepthDataset(Dataset):
    def __init__(self, root: Path, img_size=128, max_samples=None):
        self.root = Path(root)
        self.img_size = img_size

        self.rgb_files = sorted(self.root.glob("*_rgb.png"))
        if max_samples is not None:
            self.rgb_files = self.rgb_files[:max_samples]

        assert len(self.rgb_files) > 0, f"No *_rgb.png files found in {self.root}"

    def __len__(self):
        return len(self.rgb_files)

    def __getitem__(self, idx):
        rgb_path = self.rgb_files[idx]
        depth_path = Path(str(rgb_path).replace("_rgb.png", "_depth.npy"))

        rgb = np.array(Image.open(rgb_path).convert("RGB"), dtype=np.float32) / 255.0
        depth = np.load(depth_path).astype(np.float32)

        rgb_t = torch.from_numpy(rgb).permute(2, 0, 1).unsqueeze(0)
        rgb_t = F.interpolate(rgb_t, size=(self.img_size, self.img_size), mode="bilinear", align_corners=False)
        rgb_t = rgb_t.squeeze(0)

        depth_t = torch.from_numpy(depth).unsqueeze(0).unsqueeze(0)
        depth_t = F.interpolate(depth_t, size=(self.img_size, self.img_size), mode="nearest")
        depth_t = depth_t.squeeze(0)

        valid_mask = (depth_t > 0).float()
        depth_t = torch.clamp(depth_t, min=1e-3, max=80.0)
        depth_log_t = torch.log(depth_t)

        return {
            "image": rgb_t,
            "depth": depth_t,
            "depth_log": depth_log_t,
            "mask": valid_mask,
            "name": rgb_path.name,
        }

In [5]:
full_dataset = SimpleDepthDataset(DATA_ROOT, img_size=IMG_SIZE, max_samples=MAX_TRAIN_SAMPLES + MAX_VAL_SAMPLES)

n_total = len(full_dataset)
n_val = min(MAX_VAL_SAMPLES, max(1, int(0.15 * n_total)))
n_train = n_total - n_val

train_dataset, val_dataset = random_split(
    full_dataset,
    [n_train, n_val],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples:   {len(val_dataset)}")

Train samples: 4080
Val samples:   720


### Create Neural Network

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        groups = max(1, min(8, out_ch // 4))
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(groups, out_ch),
            nn.SiLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(groups, out_ch),
            nn.SiLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class SmallUNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.enc1 = ConvBlock(3, 32)
        self.pool1 = nn.MaxPool2d(2)

        self.enc2 = ConvBlock(32, 64)
        self.pool2 = nn.MaxPool2d(2)

        self.enc3 = ConvBlock(64, 128)
        self.pool3 = nn.MaxPool2d(2)

        self.bottleneck = ConvBlock(128, 256)

        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec3 = ConvBlock(256, 128)

        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec2 = ConvBlock(128, 64)

        self.up1 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.dec1 = ConvBlock(64, 32)

        self.out_conv = nn.Conv2d(32, 1, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))

        b = self.bottleneck(self.pool3(e3))

        d3 = self.up3(b)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        # output is log-depth in meters (no sigmoid clipping)
        return self.out_conv(d1)


model = SmallUNet().to(DEVICE)
print(model)

TinyUNet(
  (enc1): DoubleConv(
    (net): Sequential(
      (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU(inplace=True)
      (2): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU(inplace=True)
    )
  )
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (enc2): DoubleConv(
    (net): Sequential(
      (0): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU(inplace=True)
      (2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU(inplace=True)
    )
  )
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (bottleneck): DoubleConv(
    (net): Sequential(
      (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU(inplace=True)
      (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU(inplace=True)
    )
  )
  (up2

### Loss Function

In [ ]:
def si_log_rmse_loss(pred_log, target_log, mask, eps=1e-6):
    """
    Scale-invariant RMSE surrogate in log domain.

    pred_log, target_log: [B,1,H,W] in log-depth space
    mask:                 [B,1,H,W] (1 = valid, 0 = ignore)
    """
    valid = mask > 0
    if valid.sum() == 0:
        return pred_log.new_tensor(0.0)

    d = pred_log[valid] - target_log[valid]
    si_var = torch.mean(d * d) - torch.mean(d) ** 2
    return torch.sqrt(torch.clamp(si_var, min=eps))

### train and evaluate the model

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=1,
)


def compute_depth_metrics_from_log(pred_log, target_log, mask, eps=1e-6):
    valid = mask > 0
    if valid.sum() == 0:
        return {"si_rmse": float("nan"), "abs_rel": float("nan"), "rmse_m": float("nan")}

    d = pred_log[valid] - target_log[valid]
    si_var = torch.mean(d * d) - torch.mean(d) ** 2
    si_rmse = torch.sqrt(torch.clamp(si_var, min=eps))

    pred_m = torch.exp(pred_log[valid])
    tgt_m = torch.exp(target_log[valid])
    abs_rel = torch.mean(torch.abs(pred_m - tgt_m) / torch.clamp(tgt_m, min=1e-3))
    rmse_m = torch.sqrt(torch.mean((pred_m - tgt_m) ** 2))

    return {
        "si_rmse": si_rmse.item(),
        "abs_rel": abs_rel.item(),
        "rmse_m": rmse_m.item(),
    }


def run_epoch(loader, model, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    total_si_rmse = 0.0
    total_abs_rel = 0.0
    total_rmse_m = 0.0
    n_batches = 0

    for batch in loader:
        images = batch["image"].to(DEVICE)
        depth_log = batch["depth_log"].to(DEVICE)
        masks = batch["mask"].to(DEVICE)

        if is_train:
            flip_mask = torch.rand(images.shape[0], device=images.device) < 0.5
            if flip_mask.any():
                images[flip_mask] = torch.flip(images[flip_mask], dims=[3])
                depth_log[flip_mask] = torch.flip(depth_log[flip_mask], dims=[3])
                masks[flip_mask] = torch.flip(masks[flip_mask], dims=[3])

            brightness = torch.empty(images.shape[0], 1, 1, 1, device=images.device).uniform_(0.9, 1.1)
            contrast = torch.empty(images.shape[0], 1, 1, 1, device=images.device).uniform_(0.9, 1.1)
            image_mean = images.mean(dim=(2, 3), keepdim=True)
            images = (images - image_mean) * contrast + image_mean
            images = torch.clamp(images * brightness, 0.0, 1.0)

        with torch.set_grad_enabled(is_train):
            pred_log = model(images)
            loss = si_log_rmse_loss(pred_log, depth_log, masks)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

        metrics = compute_depth_metrics_from_log(pred_log.detach(), depth_log, masks)
        total_loss += loss.item()
        total_si_rmse += metrics["si_rmse"]
        total_abs_rel += metrics["abs_rel"]
        total_rmse_m += metrics["rmse_m"]
        n_batches += 1

    return {
        "loss": total_loss / max(1, n_batches),
        "si_rmse": total_si_rmse / max(1, n_batches),
        "abs_rel": total_abs_rel / max(1, n_batches),
        "rmse_m": total_rmse_m / max(1, n_batches),
    }


for epoch in range(NUM_EPOCHS):
    train_metrics = run_epoch(train_loader, model, optimizer=optimizer)
    val_metrics = run_epoch(val_loader, model, optimizer=None)
    scheduler.step(val_metrics["loss"])

    print(
        f"Epoch {epoch+1}/{NUM_EPOCHS} | "
        f"train loss: {train_metrics['loss']:.4f}, SI-RMSE: {train_metrics['si_rmse']:.4f}, AbsRel: {train_metrics['abs_rel']:.4f}, RMSE[m]: {train_metrics['rmse_m']:.3f} | "
        f"val loss: {val_metrics['loss']:.4f}, SI-RMSE: {val_metrics['si_rmse']:.4f}, AbsRel: {val_metrics['abs_rel']:.4f}, RMSE[m]: {val_metrics['rmse_m']:.3f}"
    )

Epoch 1/8 | train loss: 1.5794, AbsRel: 8.4653, RMSE: 0.1798 | val loss: 1.4625, AbsRel: 5.5169, RMSE: 0.1834


In [ ]:
model.eval()

batch = next(iter(val_loader))
images = batch["image"].to(DEVICE)
depths_m = batch["depth"].to(DEVICE)
depths_log = batch["depth_log"].to(DEVICE)
masks = batch["mask"].to(DEVICE)
names = batch["name"]

with torch.no_grad():
    preds_log = model(images)
    preds_m = torch.exp(preds_log).clamp(min=1e-3, max=80.0)

i = 0
img = images[i].cpu().permute(1, 2, 0).numpy()
gt = depths_m[i, 0].cpu().numpy()
pred = preds_m[i, 0].cpu().numpy()
mask = masks[i, 0].cpu().numpy()

# hide invalid gt pixels
gt_vis = gt.copy()
gt_vis[mask == 0] = np.nan

valid_vals = gt_vis[~np.isnan(gt_vis)]
if valid_vals.size > 0:
    vmin, vmax = np.percentile(valid_vals, [1, 99])
else:
    vmin, vmax = 0.0, 1.0

pred_vis = np.clip(pred, vmin, vmax)

sample_metrics = compute_depth_metrics_from_log(
    preds_log[i:i+1], depths_log[i:i+1], masks[i:i+1]
)

plt.figure(figsize=(14, 4))

plt.subplot(1, 3, 1)
plt.imshow(img)
plt.title("RGB")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(gt_vis, cmap="viridis", vmin=vmin, vmax=vmax)
plt.title("Ground Truth Depth [m]")
plt.axis("off")
plt.colorbar(fraction=0.046, pad=0.04)

plt.subplot(1, 3, 3)
plt.imshow(pred_vis, cmap="viridis", vmin=vmin, vmax=vmax)
plt.title("Predicted Depth [m]")
plt.axis("off")
plt.colorbar(fraction=0.046, pad=0.04)

plt.suptitle(f"{names[i]} | SI-RMSE={sample_metrics['si_rmse']:.4f}")
plt.tight_layout()
plt.show()